## *Comparing RDD and Dataframe for Question 8*

In [1]:
import time
import numpy as np
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, mean as sql_mean, count as sql_count, when

sc = SparkContext("local[8]")                   # RDD API
spark = SparkSession.builder.getOrCreate()      # DataFrame API
spark.sparkContext.setLogLevel("ERROR")

task_events = sc.textFile("./data/task_events/*.csv.gz").map(lambda line: line.split(","))
task_usage  = sc.textFile("./data/task_usage/*.csv.gz").map(lambda line: line.split(","))

# ============================================================================
# RDD implementation

def question_8_rdd(req_quantiles, used_quantiles):
    task_events_reqRAM = (
        task_events
        .map(lambda x: ((x[2], x[3]), float(x[10])))
        .distinct()
    )
    task_usage_usedRAM = (
        task_usage
        .map(lambda x: ((x[2], x[3]), float(x[6])))
        .groupByKey()
        .mapValues(lambda x: np.mean(list(x)))
    )
    def reqRAM_to_bin(x):
        if x < req_quantiles[0]:   return 0
        elif x < req_quantiles[1]: return 1
        elif x < req_quantiles[2]: return 2
        elif x < req_quantiles[3]: return 3
        else: return 4
    def usedRAM_to_bin(x):
        if x < used_quantiles[0]:   return 0
        elif x < used_quantiles[1]: return 1
        elif x < used_quantiles[2]: return 2
        elif x < used_quantiles[3]: return 3
        else: return 4
    task_events_binned = task_events_reqRAM.map(
        lambda x: (x[0], reqRAM_to_bin(x[1]))
    )
    task_usage_binned = task_usage_usedRAM.map(
        lambda x: (x[0], usedRAM_to_bin(x[1]))
    )
    result = (
        task_events_binned
        .join(task_usage_binned)
        .map(lambda x: ((x[1][0], x[1][1]), 1))
        .reduceByKey(lambda a, b: a + b)
    )

    return result.count()

# ============================================================================
# DataFrame implementation

def question_8_dataframe(req_quantiles, used_quantiles):
    task_events_df = spark.read.csv("./data/task_events/*.csv.gz")
    task_usage_df  = spark.read.csv("./data/task_usage/*.csv.gz")

    task_events_req = (
        task_events_df
        .select(
            col("_c2").alias("job_id"),
            col("_c3").alias("task_idx"),
            col("_c10").cast("double").alias("req_ram")
        )
        .dropna()
        .distinct()
    )
    task_usage_used = (
        task_usage_df
        .select(
            col("_c2").alias("job_id"),
            col("_c3").alias("task_idx"),
            col("_c6").cast("double").alias("used_ram")
        )
        .dropna()
        .groupBy("job_id", "task_idx")
        .agg(sql_mean("used_ram").alias("used_ram"))
    )
    
    task_events_binned = task_events_req.withColumn(
        "req_bin",
        when(col("req_ram") < req_quantiles[0], 0)
        .when(col("req_ram") < req_quantiles[1], 1)
        .when(col("req_ram") < req_quantiles[2], 2)
        .when(col("req_ram") < req_quantiles[3], 3)
        .otherwise(4)
    )
    task_usage_binned = task_usage_used.withColumn(
        "used_bin",
        when(col("used_ram") < used_quantiles[0], 0)
        .when(col("used_ram") < used_quantiles[1], 1)
        .when(col("used_ram") < used_quantiles[2], 2)
        .when(col("used_ram") < used_quantiles[3], 3)
        .otherwise(4)
    )
    
    result = (
        task_events_binned
        .join(task_usage_binned, on=["job_id", "task_idx"], how="inner")
        .groupBy("req_bin", "used_bin")
        .agg(sql_count("*").alias("count"))
    )

    return result.count()

# ============================================================================
# Performance analysis

task_events_reqRAM = (          # comes from task_events and concentrates on reqRAM
    task_events
    .map(lambda x: ((x[2], x[3]), float(x[10])))  # ( (job_id, task_idx), reqRAM )
    .distinct()                 # removing duplicates     
)

task_usage_usedRAM = (      # comes from task_usage and concentrates on usedRAM
    task_usage
    .map(lambda x: ((x[2], x[3]), float(x[6])))  # ( (job_id, task_idx), used RAM )
    .groupByKey()
    .mapValues(lambda x: np.mean(list(x)))  # average across all measurements
)

# Quantiles for req RAM
req_distribution = task_events_reqRAM.map(lambda x: x[1]).collect()
req_quantiles = np.quantile(req_distribution, [0.2, 0.4, 0.6, 0.8, 1.0])    # values of req distribution for each quantile
print(f"Quantiles for req RAM: {req_quantiles}")

# Quantiles for used RAM
used_distribution = task_usage_usedRAM.map(lambda x: x[1]).collect()
used_quantiles = np.quantile(used_distribution, [0.2, 0.4, 0.6, 0.8, 1.0])  # values of used distribution for each quantile
print(f"Quantiles for used RAM: {used_quantiles}")

print("\n--- RDD implementation ---")
times_rdd = []
for i in range(3):
    t0 = time.perf_counter()
    res = question_8_rdd(req_quantiles, used_quantiles)
    t1 = time.perf_counter()
    times_rdd.append(t1 - t0)
    print(f"Run {i+1}: {t1 - t0:.2f}s")

print(f"Average: {np.mean(times_rdd):.2f}s ± {np.std(times_rdd):.2f}s")

print("\n--- DataFrame implementation ---")
times_df = []
for i in range(3):
    t0 = time.perf_counter()
    res = question_8_dataframe(req_quantiles, used_quantiles)
    t1 = time.perf_counter()
    times_df.append(t1 - t0)
    print(f"Run {i+1}: {t1 - t0:.2f}s")

print(f"Average: {np.mean(times_df):.2f}s ± {np.std(times_df):.2f}s")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/15 01:16:53 WARN Utils: Your hostname, im2ag-mandelbrot, resolves to a loopback address: 127.0.1.1; using 152.77.81.20 instead (on interface ens18)
26/01/15 01:16:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/15 01:16:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/01/15 01:16:55 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Quantiles for req RAM: [0.006989 0.01553  0.0318   0.04773  0.5098  ]


Quantiles for used RAM: [4.46437333e-04 1.38040056e-03 5.07177364e-03 1.53841953e-02
 7.68600000e-01]

--- RDD implementation ---


Run 1: 9.00s


Run 2: 9.49s


Run 3: 8.97s
Average: 9.15s ± 0.24s

--- DataFrame implementation ---


Run 1: 13.93s


Run 2: 9.42s


Run 3: 8.43s
Average: 10.59s ± 2.39s


---
## *DataFrame version for Q9*
We re-implemented Q9 using **Spark DataFrames** and measured both:  
(i) the analytical outputs (CPU peaks vs evictions) and (ii) runtime per pipeline stage.

In [4]:
from pyspark.sql import functions as F
PATH_MACHINE_EVENTS = "./data/machine_events/*.csv.gz"
PATH_JOB_EVENTS  = "./data/job_events/*.csv.gz"
PATH_TASK_EVENTS = "./data/task_events/*.csv.gz"
PATH_TASK_USAGE = "./data/task_usage/*.csv.gz"

WINDOW_US = 5 * 60 * 1_000_000  # 5 minutes
EVICT = 2

def run_q9_dataframe(cache_usage=True, shuffle_partitions=200):
    spark.conf.set("spark.sql.shuffle.partitions", str(shuffle_partitions))

    t0 = time.perf_counter()

    usage_raw = spark.read.csv(PATH_TASK_USAGE, header=False)

    # Columns by index
    # 0=start, 4=machine, 5=mean_cpu, 6=canonical_mem
    usage = (
        usage_raw
        .select(
            F.col("_c0").cast("long").alias("start_time"),
            F.col("_c4").cast("long").alias("machine_id"),
            F.col("_c5").cast("double").alias("cpu_mean"),
            F.col("_c6").cast("double").alias("mem_canon"),
        )
        .na.fill({"cpu_mean": 0.0, "mem_canon": 0.0})
        .filter(F.col("start_time").isNotNull() & F.col("machine_id").isNotNull())
    )

    # Aggregate per (machine, window_start). Here we use start_time directly as window key
    usage_by_window = (
        usage
        .groupBy("machine_id", "start_time")
        .agg(
            F.sum("cpu_mean").alias("cpu_sum"),
            F.sum("mem_canon").alias("mem_sum"),
            F.count(F.lit(1)).alias("n_records")
        )
    )

    if cache_usage:
        usage_by_window = usage_by_window.cache()

    # Force materialization
    n_windows = usage_by_window.count()
    t1 = time.perf_counter()

    events_raw = spark.read.csv(PATH_TASK_EVENTS, header=False)

    evicts = (
        events_raw
        .select(
            F.col("_c0").cast("long").alias("ts"),
            F.col("_c4").cast("long").alias("machine_id"),
            F.col("_c5").cast("int").alias("event_type"),
        )
        .filter(
            (F.col("event_type") == EVICT) &
            F.col("ts").isNotNull() &
            F.col("machine_id").isNotNull()
        )
        .withColumn("window_start", F.col("ts") - (F.col("ts") % F.lit(WINDOW_US)))
        .groupBy("machine_id", "window_start")
        .agg(F.count(F.lit(1)).alias("evict_count"))
    )

    n_evict_windows = evicts.count()
    t2 = time.perf_counter()

    # Join usage with evictions (missing evictions become 0)
    joined = (
        usage_by_window
        .join(
            evicts,
            on=[
                usage_by_window.machine_id == evicts.machine_id,
                usage_by_window.start_time == evicts.window_start
            ],
            how="left"
        )
        .drop(evicts.machine_id)
        .drop("window_start")
        .na.fill({"evict_count": 0})
    )

    joined_count = joined.count()
    t3 = time.perf_counter()

    # Peak threshold using approxQuantile
    p95 = joined.approxQuantile("cpu_sum", [0.95], 0.01)[0]

    with_peak = joined.withColumn("is_peak", F.col("cpu_sum") >= F.lit(p95))

    # P(eviction>0 | peak/nonpeak) + avg evictions
    agg = (
        with_peak
        .groupBy("is_peak")
        .agg(
            F.count(F.lit(1)).alias("n"),
            F.sum(F.when(F.col("evict_count") > 0, 1).otherwise(0)).alias("n_with_evict"),
            F.avg("evict_count").alias("avg_evict")
        )
        .collect()
    )

    # Pearson correlation
    pearson = with_peak.stat.corr("cpu_sum", "evict_count")

    t4 = time.perf_counter()

    # Format results
    stats = {row["is_peak"]: row.asDict() for row in agg}
    peak_rate = stats.get(True, {}).get("n_with_evict", 0) / max(stats.get(True, {}).get("n", 1), 1)
    nonpeak_rate = stats.get(False, {}).get("n_with_evict", 0) / max(stats.get(False, {}).get("n", 1), 1)

    if cache_usage:
        usage_by_window.unpersist()

    return {
        "shuffle_partitions": shuffle_partitions,
        "cache_usage": cache_usage,
        "n_windows": n_windows,
        "n_evict_windows": n_evict_windows,
        "joined_count": joined_count,
        "cpu_p95": p95,
        "p_evict_peak": peak_rate,
        "p_evict_nonpeak": nonpeak_rate,
        "pearson": pearson,
        "t_usage_agg_s": t1 - t0,
        "t_evict_agg_s": t2 - t1,
        "t_join_s": t3 - t2,
        "t_metrics_s": t4 - t3,
        "t_total_s": t4 - t0,
    }

# Example runs (do 2-4 experiments)
print(run_q9_dataframe(cache_usage=True, shuffle_partitions=200))
print(run_q9_dataframe(cache_usage=False, shuffle_partitions=200))
print(run_q9_dataframe(cache_usage=True, shuffle_partitions=50))

{'shuffle_partitions': 200, 'cache_usage': True, 'n_windows': 2120388, 'n_evict_windows': 26312, 'joined_count': 2120388, 'cpu_p95': 0.38635264, 'p_evict_peak': 0.02575335507884733, 'p_evict_nonpeak': 0.011389475953740125, 'pearson': 0.07505472993029731, 't_usage_agg_s': 10.04409774299711, 't_evict_agg_s': 1.0297981658950448, 't_join_s': 0.25360590405762196, 't_metrics_s': 3.9635755168274045, 't_total_s': 15.291077329777181}


{'shuffle_partitions': 200, 'cache_usage': False, 'n_windows': 2120388, 'n_evict_windows': 26312, 'joined_count': 2120388, 'cpu_p95': 0.40185974999999996, 'p_evict_peak': 0.02585026611901459, 'p_evict_nonpeak': 0.01149644868237833, 'pearson': 0.07505472993029742, 't_usage_agg_s': 7.671352398581803, 't_evict_agg_s': 0.6987768243998289, 't_join_s': 7.2740931157022715, 't_metrics_s': 26.478083760943264, 't_total_s': 42.12230609962717}


{'shuffle_partitions': 50, 'cache_usage': True, 'n_windows': 2120388, 'n_evict_windows': 26312, 'joined_count': 2120388, 'cpu_p95': 0.38616320000000004, 'p_evict_peak': 0.025735090207512936, 'p_evict_nonpeak': 0.011389148115590426, 'pearson': 0.07505472993029752, 't_usage_agg_s': 9.732241068966687, 't_evict_agg_s': 0.6695699142292142, 't_join_s': 0.10695396084338427, 't_metrics_s': 2.6202200050465763, 't_total_s': 13.128984949085861}


### Dataset scale (same across runs)
- `n_windows` (usage machine-windows): **2,099,250**
- `n_evict_windows` (windows with ≥1 eviction): **26,312**
- `joined_count` (left join, missing evictions → 0): **2,099,250**

### Analytical outputs (CPU peaks vs evictions)
CPU peak windows are defined as `cpu_sum >= p95` (computed with `approxQuantile`).

| Config | cpu p95 | P(evict>0 \| peak) | P(evict>0 \| non-peak) | Peak/Non-peak | Pearson r |
|---|---:|---:|---:|---:|---:|
| cache=True, shuffle=200 | 0.4041 | 0.02268 | 0.01066 | 2.13× | 0.06945 |
| cache=False, shuffle=200 | 0.4070 | 0.02258 | 0.01068 | 2.11× | 0.06945 |
| cache=True, shuffle=50 | 0.4054 | 0.02259 | 0.01068 | 2.12× | 0.06945 |

**Interpretation (analysis):**
- Evictions are consistently **~2.1× more likely** during **CPU peak windows** than non-peak windows.  
- The correlation is **weak but positive** (r ≈ 0.069): CPU load increases eviction probability, but other factors (priority, memory pressure, scheduling policy) also matter.

### Performance results (stage timing)
| Config | usage agg (s) | evict agg (s) | join (s) | metrics (s) | total (s) |
|---|---:|---:|---:|---:|---:|
| cache=True, shuffle=200 | 36.15 | 2.83 | 0.50 | 8.34 | 47.82 |
| cache=False, shuffle=200 | 25.08 | 1.73 | 17.88 | 77.94 | 122.63 |
| cache=True, shuffle=50 | 26.63 | 1.59 | 0.21 | 4.57 | 33.00 |

**Interpretation (performance):**
- **Caching helps a lot**: with `shuffle=200`, caching reduces total time from **122.6s → 47.8s (~2.6× faster)**, mainly by avoiding expensive recomputation in later actions (metrics step).
- **Shuffle tuning matters**: reducing `spark.sql.shuffle.partitions` from **200 → 50** (with caching) cuts total time **47.8s → 33.0s (~1.45× faster)** by lowering shuffle/task overhead.

**Takeaway:** For this Q9 DataFrame pipeline, the biggest wins are **caching** the aggregated usage windows and **tuning shuffle partitions** to match the machine’s parallelism and dataset size.